In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
PROJECT_ROOT = "/content/drive/MyDrive/KCA_Hallucination_Reduction"

%cd $PROJECT_ROOT

/content/drive/MyDrive/KCA_Hallucination_Reduction


In [3]:
import os

print(os.listdir(PROJECT_ROOT))
print(os.listdir(PROJECT_ROOT + "/src"))

['README.md', 'requirements.txt', 'config.yaml', 'train.py', 'inference.py', '.gitignore', 'results', 'src', 'notebooks', 'data', 'checkpoints']
['dataset.py', 'model.py', 'utils.py', '__pycache__']


In [4]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d332cf5fe403b1b4dd5184ba6acf576831c0b2f4167ff6d8edeb862d70b029e3
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c

In [5]:
import sys
sys.path.append(PROJECT_ROOT)

from src.model import load_finetuned_model, generate_answer
from src.utils import load_config

In [6]:
config = load_config("config.yaml")
config

{'project_name': 'KCA Hallucination Reduction',
 'base_model': 'NousResearch/Llama-2-7b-hf',
 'training': {'strategy': 'refusal',
  'train_file': 'data/sample_data.csv',
  'output_dir': 'checkpoints/model_refusal',
  'max_seq_length': 1024,
  'num_train_epochs': 1,
  'per_device_train_batch_size': 4,
  'gradient_accumulation_steps': 4,
  'learning_rate': 0.0002,
  'warmup_ratio': 0.03,
  'logging_steps': 20,
  'save_steps': 200,
  'samples_per_class': 500},
 'lora': {'r': 16,
  'lora_alpha': 32,
  'lora_dropout': 0.05,
  'target_modules': ['q_proj',
   'k_proj',
   'v_proj',
   'o_proj',
   'gate_proj',
   'up_proj',
   'down_proj']},
 'generation': {'max_new_tokens': 256,
  'temperature': 0.3,
  'top_p': 0.9,
  'refusal_aware': True}}

In [8]:
from huggingface_hub import login

login()

In [9]:
model, tokenizer = load_finetuned_model(
    base_model_name=config["base_model"],
    adapter_dir="checkpoints/model_refusal"
)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [10]:
prompt = "Who wrote Hamlet?"

answer = generate_answer(
    model=model,
    tokenizer=tokenizer,
    instruction=prompt,
    max_new_tokens=128,
    temperature=0.3,
    top_p=0.9,
    refusal_aware=True
)

print(answer)

Hamlet was written by William Shakespeare, a famous English playwright and poet. He is considered one of the greatest writers in the English language and is widely regarded as the greatest dramatist of all time. Shakespeare wrote many other famous plays, including Romeo and Juliet, Macbeth, and Othello.
